# Follow-up training audit
Safe aggregate reconciliation; case review is qualitative and does not replace original grades.


In [ ]:
import json, math
from pathlib import Path
p=Path('docs') if Path('docs').is_dir() else Path('.')
s=json.loads((p/'targeted_book_pilot_record_audit_20260908.safe.json').read_text(encoding='utf-8'))
assert s['exact_retokenized_records']==80
assert s['supervised_tokens_revalidated']==7183
assert s['sequence_tokens_reconciled']==15491
assert [r['step'] for r in s['steps']]==list(range(1,11))
assert all(math.isfinite(v) for r in s['steps'] for v in r.values())
assert sum(r['train/global_tokens'] for r in s['steps'])==15491
assert all(r['train/lr']==5e-7 for r in s['steps'])
length=s['closed_answer_lengths']
assert all(r['count']==110 for r in length.values())
assert length['baseline']['mean']>length['step5']['mean']>length['step10']['mean']
print('PASS: supervision, complete step logs, token totals, answer-length cohorts.')


In [ ]:
g=json.loads((p/'targeted_book_train64_grading_20260908.safe.json').read_text(encoding='utf-8'))
assert g['items']==64 and g['status']=={'invalid_review':5,'scored':59}
c=g['cohorts']['in_training_opportunities']
assert c['items']==64 and c['scored']==59
assert [c['metrics'][m]['closed']['both_pass'] for m in ('baseline','step5','step10')]==[2,1,0]
assert [c['metrics'][m]['open']['both_pass'] for m in ('baseline','step5','step10')]==[58,58,57]
for model,counts in c['paired_closed_both'].items():
    assert sum(counts.values())==59
    assert counts.get('gain',0)-counts.get('loss',0)==c['metrics'][model]['closed']['both_pass']-2
assert g['usage']['total_tokens']==462829
print('PASS: training-only cohort, common valid denominator and paired changes.')
